<a href="https://colab.research.google.com/github/akatsushikami-arch/HW/blob/main/%D0%9F%D0%BE%D1%80%D0%BE%D0%B4%D1%8B_%D1%81%D0%BE%D0%B1%D0%B0%D0%BA%2C_%D0%BF%D1%80%D0%BE%D1%81%D1%82%D0%B0%D1%8F_%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D1%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models

In [6]:
!pip install importlib_resources

In [7]:
dataset, info = tfds.load('oxford_iiit_pet', with_info=True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.9HGPS9_4.0.0/oxford_iiit_pet-train.tfrecord-[0-…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.9HGPS9_4.0.0/oxford_iiit_pet-test.tfrecord-[0-9…

Dataset oxford_iiit_pet downloaded and prepared to /root/tensorflow_datasets/oxford_iiit_pet/4.0.0. Subsequent calls will reuse this data.


In [8]:
# собаки с меткой 1
def is_dog(sample):
    return sample['species'] == 1

In [9]:
train_dogs = dataset['train'].filter(is_dog)
test_dogs = dataset['test'].filter(is_dog)

In [10]:
IMG_SIZE = 128
BATCH_SIZE = 32

In [11]:
# Метки собак от 0 до 24
dog_labels = [i for i, name in enumerate(info.features['label'].names) if name[0].islower()]
lookup_table = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(dog_labels, dtype=tf.int64),
        values=tf.constant(list(range(len(dog_labels))), dtype=tf.int64)
         ),
    default_value=-1
    )

In [12]:
def preprocess(sample):
    image = tf.image.resize(sample['image'], (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    label = lookup_table.lookup(sample['label'])
    return image, label

In [13]:
train_ds = train_dogs.map(preprocess).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_dogs.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [14]:
NUM_CLASSES = len(dog_labels) # 25 пород собак

In [15]:
# архитектура CNN
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_layer')

In [16]:
# простые границы и пятна
x = layers.Conv2D(filters=32, kernel_size=(3, 3), padding='same', activation='relu')(inputs)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)

In [17]:
# элементы морды и ушей
x = layers.Conv2D(filters=64, kernel_size=(3, 3), padding='same', activation='relu')(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)

In [18]:
# признаки породы
x = layers.Conv2D(filters=128, kernel_size=(3, 3), padding='same', activation='relu')(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)

In [19]:
# Переход к классификации
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='output_layer')(x)

In [20]:
model = models.Model(inputs=inputs, outputs=outputs, name='dog_breed_cnn')
model.summary()

Model: "dog_breed_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 25)             │         3,225 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 96,473 (376.85 KB)

 Trainable params: 96,473 (376.85 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
# обучение модели
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [22]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15
)

Epoch 1/15
     78/Unknown 94s 1s/step - accuracy: 0.0325 - loss: 3.2216

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


78/78 ━━━━━━━━━━━━━━━━━━━━ 126s 2s/step - accuracy: 0.0357 - loss: 3.2203 - val_accuracy: 0.0406 - val_loss: 3.2169
Epoch 2/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 139s 2s/step - accuracy: 0.0522 - loss: 3.2115 - val_accuracy: 0.0499 - val_loss: 3.2044
Epoch 3/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 142s 2s/step - accuracy: 0.0642 - loss: 3.1831 - val_accuracy: 0.0571 - val_loss: 3.2033
Epoch 4/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 142s 2s/step - accuracy: 0.0678 - loss: 3.1641 - val_accuracy: 0.0603 - val_loss: 3.2067
Epoch 5/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 143s 2s/step - accuracy: 0.0787 - loss: 3.1505 - val_accuracy: 0.0772 - val_loss: 3.1750
Epoch 6/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 200s 2s/step - accuracy: 0.0803 - loss: 3.1289 - val_accuracy: 0.0632 - val_loss: 3.2179
Epoch 7/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 122s 1s/step - accuracy: 0.0851 - loss: 3.1117 - val_accuracy: 0.0619 - val_loss: 3.2426
Epoch 8/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 128s 2s/step - accuracy: 0.0947 - loss: 3.0940 - val_accuracy: 0.0760 - val_loss: 3.